# v11 E (GPU 1): a SECOND SEED of the related run (Hindi -> Gujarati -> Punjabi), 21 phases (7 cycles)
Exactly the recipe of `train_v8_INDIC_CONT_gpu0.ipynb`, with seed 1 (different random start, data order and dropout). Run name `v8_indic_continual_s1`.
Takes more than a day. Resumes after the last saved phase if restarted. Every finished phase is backed up to Hugging Face.
Progress: `tail -3 runs/v8_indic_continual_s1/log.jsonl`

In [1]:
import os, sys, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["CML_LANGSET"] = "indic"             # Hindi, Gujarati, Punjabi + tokenizer_v8 (must be set before importing common_v8)
os.environ["HF_HUB_DISABLE_XET"] = "1"
REPO = "/workspace/shreshth"; os.chdir(REPO); sys.path.insert(0, REPO)
subprocess.run([sys.executable, "-u", "prep_indic_data.py"], check=True)

/venv/main/lib/python3.12/site-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Hindi training pairs: 993,016 -> Gujarati and Punjabi get exactly the same number
  hi: 1.37 pieces per word (2,000 validation sentences)
  gu: 1.52 pieces per word (2,000 validation sentences)
  pa: 1.31 pieces per word (2,000 validation sentences)


data ready: {'hi': 993016, 'gu': 993016, 'pa': 993016}


CompletedProcess(args=['/venv/main/bin/python', '-u', 'prep_indic_data.py'], returncode=0)

In [2]:
import glob, json, time
from common_v8 import *
print("languages:", LANGS, "| tags:", LANG_TAG, "| vocab:", V, "| device:", DEVICE)
print("pieces per script:", {k: sum(1 for x in CLS if x == k) for k in ["hi", "gu", "pa", "latin", "common"]})
RUN = "v8_indic_continual_s1"
from huggingface_hub import HfApi
HF_REPO = HfApi().whoami()["name"] + "/cml-lang-self-checkpoints"

languages: ['hi', 'gu', 'pa'] | tags: {'hi': 4, 'gu': 5, 'pa': 6} | vocab: 32000 | device: cuda
pieces per script: {'hi': 7550, 'gu': 7768, 'pa': 7248, 'latin': 8427, 'common': 1006}


/venv/main/lib/python3.12/site-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


In [3]:
C = RunConfig(run_name=RUN, mode="continual", order=("hi", "gu", "pa"), n_phases=21, seed=1, time_cap_h=120.0, hf_repo=HF_REPO)
print(C)

RunConfig(run_name='v8_indic_continual_s1', mode='continual', order=('hi', 'gu', 'pa'), init_ckpt=None, n_phases=21, seed=1, max_tokens=16000, lr=0.0002, min_lr=2e-05, warmup_steps=500, cooldown_steps=2000, min_steps=2000, max_stable_steps=60000, eval_every=500, plateau_window=4, plateau_tol=0.005, weight_decay=0.1, grad_clip=1.0, masked=True, diverge_factor=1.15, max_retries=2, time_cap_h=120.0, hf_repo='s8n29/cml-lang-self-checkpoints', start_phase=None, noise_per_step=0.0, boundary_push=0.0, health_every=1, redo_every=0, redo_thresh=0.01)


In [4]:
D = load_data(train_langs=["hi", "gu", "pa"])

  hi: 993,016 train / 2,000 val


  gu: 993,016 train / 2,000 val


  pa: 993,016 train / 2,000 val
data loaded in 40s


In [5]:
model = train_run(C, D)

 RUN v8_indic_continual_s1 | mode continual | 21 phases after phase 0 | device cuda
 phase order: init (random) → hi → gu → pa → hi → gu → pa → hi → ...
 each phase: fresh AdamW (wd 0.1 on matrices), warmup 500 → lr 0.0002 → plateau (<0.5% over 4-eval windows, after 2,000, cap 60,000) → cooldown 2,000 steps to 2e-05
 masked (script-restricted) loss & decoding: True | divergence guard: >1.15x best twice → retry at half lr (max 2)


phase 0: fresh random model (66.8M params)



 PHASE 0 DONE | trained: init | starting model, no training
--------------------------------------------------------------------------------------
   lang      val       Δ |   chrF      Δ |  chrF (unrestricted)  val (unrestr.)
   hi      9.143         |    2.0        |                  0.4          10.457
   gu      9.153         |    2.2        |                  0.5          10.447
   pa      9.181         |    2.1        |                  0.2          10.533

▶ PHASE 1/21 | training hi (993,016 pairs)


[W1116 09:56:58.865648385 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1929379840 bytes (free: 358350848, total: 25252724736).


p000_init.pt:   0%|          | 0.00/267M [00:00<?, ?B/s]

  [p01 hi] step    500 | lr 2.0e-04 | train 6.783 | val 5.198 (best 5.198) | window Δ n/a | stable | 80k tok/s | phase 1m33s | run ETA: after first phase | health {'wnorm': 142.442, 'rank_enc1': 14.5, 'rank_dec1': 28.9, 'rarely_active': 0.0013}


... (554 outputs left out)

  [p21 pa] step  7,000 | lr 1.1e-04 | train 2.609 | val 1.848 (best 1.848) | window Δ +0.69% | cooldown (plateau, ends at 8,000) | 76k tok/s | phase 21m22s | run ETA ~36m55s | health {'wnorm': 230.601, 'rank_enc1': 993.0, 'rank_dec1': 682.4, 'rarely_active': 0.0404}


  [p21 pa] step  7,500 | lr 6.5e-05 | train 2.529 | val 1.829 (best 1.829) | window Δ +1.17% | cooldown (plateau, ends at 8,000) | 78k tok/s | phase 22m53s | run ETA ~36m55s | health {'wnorm': 230.077, 'rank_enc1': 997.7, 'rank_dec1': 659.0, 'rarely_active': 0.0413}


  phase 21 finished (plateau); evaluating all languages...


  new best checkpoint: pa = phase 21 (chrF 45.4)

 PHASE 21 DONE | trained: pa | 8,000 steps | stop: plateau | peak lr 2.0e-04 | took 24m37s | run time 12h43m
--------------------------------------------------------------------------------------
   lang      val       Δ |   chrF      Δ |  chrF (unrestricted)  val (unrestr.)
   hi      3.105  +0.124 |   39.6   +1.2 |                  0.0           6.526
   gu      3.112  +1.135 |   38.8   -6.4 |                  0.0           5.979
   pa      1.817  -1.883 |   45.4  +12.7 |                  2.1           3.601  ◀ trained

 SUMMARY v8_indic_continual_s1: 22 phases (incl. phase 0)
--------------------------------------------------------------------------------------
  ph lang   steps      stop | chrF hi chrF gu chrF pa | val hi val gu val pa   (script-restricted)
   0 init       0      init |     2.0     2.2     2.1 |  9.143  9.153  9.181
   1   hi  28,500   plateau |    49.2     0.5     0.5 |  1.414  8.337  8.494
   2   gu  13,500   plat

In [6]:
print_run_summary(RUN)

 SUMMARY v8_indic_continual_s1: 22 phases (incl. phase 0)
--------------------------------------------------------------------------------------
  ph lang   steps      stop | chrF hi chrF gu chrF pa | val hi val gu val pa   (script-restricted)
   0 init       0      init |     2.0     2.2     2.1 |  9.143  9.153  9.181
   1   hi  28,500   plateau |    49.2     0.5     0.5 |  1.414  8.337  8.494
   2   gu  13,500   plateau |    25.9    43.9     0.5 |  4.252  2.054  8.816
   3   pa  15,000   plateau |    11.4    29.9    44.8 |  5.102  3.946  1.869
   4   hi  20,500   plateau |    51.1    16.4    31.9 |  1.337  5.401  4.489
   5   gu  11,000   plateau |    34.8    44.2    16.5 |  3.401  2.003  4.747
   6   pa  10,500   plateau |    32.0    35.4    44.8 |  3.708  3.341  1.838
   7   hi  13,500   plateau |    52.0    30.6    39.8 |  1.321  4.481  3.630
   8   gu   8,500   plateau |    37.9    44.6    27.8 |  3.083  1.990  3.909
   9   pa   9,500   plateau |    36.2    37.5    45.3 |  3.357 